In [ ]:
import numpy as np
import pandas as pd
import os
import time
import pprint
import itertools

from math import ceil, sqrt

from sklearn.base import is_regressor
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score, confusion_matrix, log_loss
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.dummy import DummyClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier, NearestNeighbors
from sklearn.neural_network import MLPClassifier
from joblib import Parallel, delayed

import pyagrum as gum
from pyagrum.skbn import BNClassifier
from pyagrum.lib.discreteTypeProcessor import DiscreteTypeProcessor

In [6]:
train = pd.read_parquet(os.path.join("..", "out", "processed", "train.parquet"))
val = pd.read_parquet(os.path.join("..", "out", "processed", "val.parquet"))
test = pd.read_parquet(os.path.join("..", "out", "processed", "test.parquet"))

meta_val = val[["open_time"]].copy()
meta_test = test[["open_time"]].copy()

train = train.drop(columns=["open_time"])
val = val.drop(columns=["open_time"])
test = test.drop(columns=["open_time"])

X_train = train.drop(columns=["label"])
y_train = train["label"]

X_val = val.drop(columns=["label"])
y_val = val["label"]

X_test = test.drop(columns=["label"])
y_test = test["label"]

rng = np.random.RandomState(0)
train_idx = rng.choice(len(X_train), 200_000, replace=False)
valid_idx = rng.choice(len(X_val), 30_000, replace=False)

X_train_sample = X_train.iloc[train_idx].to_numpy()
y_train_sample = y_train.iloc[train_idx].to_numpy()
X_val_sample = X_val.iloc[valid_idx].to_numpy()
y_val_sample = y_val.iloc[valid_idx].to_numpy()

print("Training set shape:", X_train.shape, y_train.shape)
print("Validation set shape:", X_val.shape, y_val.shape)
print("Test set shape:", X_test.shape, y_test.shape)

print(X_train.describe().loc[["mean", "std"]].T)

Training set shape: (1127670, 36) (1127670,)
Validation set shape: (188886, 36) (188886,)
Test set shape: (281294, 36) (281294,)
                               mean       std
symbol=BTCUSDT         8.404675e-02  0.277458
symbol=ETHUSDT         9.571062e-02  0.294194
symbol=BNBUSDT         9.720840e-02  0.296241
symbol=XRPUSDT         1.029530e-01  0.303898
symbol=ADAUSDT         1.097883e-01  0.312626
symbol=DOGEUSDT        9.485488e-02  0.293015
symbol=SOLUSDT         1.016264e-01  0.302156
symbol=LTCUSDT         1.044862e-01  0.305890
symbol=LINKUSDT        1.138782e-01  0.317663
symbol=AVAXUSDT        9.544725e-02  0.293832
returns                4.839153e-18  1.000000
volatility            -2.580882e-17  1.000000
upper_wick            -6.452205e-17  1.000000
lower_wick             0.000000e+00  1.000000
body_ratio             4.375401e-17  1.000000
taker_buy_ratio       -2.790579e-16  1.000000
taker_buy_ratio_16    -1.451746e-15  1.000000
relative_volume       -1.729997e-16  1.0000

In [7]:
model_results = []

def get_scores(model, X):
    if hasattr(model, "predict_proba"):
        return model.predict_proba(X)[:, 1]
    elif hasattr(model, "decision_function"):
        return model.decision_function(X)
    return model.predict(X)

def evaluate_model(name, model, X_train, y_train, X_val, y_val):
    t0 = time.perf_counter()
    model.fit(X_train, y_train)
    fit_time = time.perf_counter() - t0

    t0 = time.perf_counter()
    raw = model.predict(X_val)
    predict_time = time.perf_counter() - t0

    reg = is_regressor(model)
    y_pred = (raw > 0.5).astype(int) if reg else raw
    scores = raw if reg else get_scores(model, X_val)

    row = {
        "model": name,
        "fit_time": fit_time,
        "predict_time": predict_time,
        "accuracy": accuracy_score(y_val, y_pred),
        "precision": precision_score(y_val, y_pred, zero_division=0),
        "recall": recall_score(y_val, y_pred, zero_division=0),
        "f1": f1_score(y_val, y_pred, zero_division=0),
        "roc_auc": roc_auc_score(y_val, scores),
        "confusion_matrix": confusion_matrix(y_val, y_pred).tolist(),
        "log_loss": log_loss(y_val, scores),
    }   
    model_results.append(row)
    pprint.pprint(row)



In [ ]:
def evaluate_logistic_regression(solver, penalty, C, max_iter=1000):
    m = LogisticRegression(
       solver=solver, 
       penalty=penalty, 
       C=C, 
       max_iter=max_iter, 
       random_state=42
    )
    m.fit(X_train, y_train)
    p_train = m.predict_proba(X_train)
    p_val = m.predict_proba(X_val)
    return {
       "solver": solver,
        "penalty": penalty,
        "C": C,
        "train_log_loss": log_loss(y_train, p_train[:, 1]),
        "val_log_loss": log_loss(y_val, p_val[:, 1]),
        "train_auc": roc_auc_score(y_train, p_train[:, 1]),
        "val_auc": roc_auc_score(y_val, p_val[:, 1])
    }

def select_logistic_regression(solvers, penalties, Cs):
    combos = list(itertools.product(solvers, penalties, Cs))
    rows = Parallel(n_jobs=-1)(delayed(evaluate_logistic_regression)(s, p, c) for s, p, c in combos)
    return pd.DataFrame(rows).sort_values(by="val_log_loss", ascending=True).reset_index(drop=True)


def evaluate_knn(n_neighbors, weights, algorithm, metric, p):
      m = KNeighborsClassifier(
         n_neighbors=n_neighbors,
         weights=weights,
         algorithm=algorithm,
         metric=metric,
         p=p,
         n_jobs=1
      )
      m.fit(X_train_sample, y_train_sample)
      p_train_sample = m.predict_proba(X_train_sample)
      p_val_sample = m.predict_proba(X_val_sample)
      print("X_train:", X_train.shape)
      print("y_train:", y_train.shape)
      print("p_train:", p_train_sample.shape)

      print("X_val:", X_val.shape)
      print("y_val:", y_val.shape)
      print("p_val:", p_val_sample.shape)
      return {
         "n_neighbors": n_neighbors,
         "weights": weights,
         "algorithm": algorithm,
         "metric": metric,
         "p": p,
         "train_log_loss": log_loss(y_train_sample, p_train_sample[:, 1]),
         "val_log_loss": log_loss(y_val_sample, p_val_sample[:, 1]),
         "train_auc": roc_auc_score(y_train_sample, p_train_sample[:, 1]),
         "val_auc": roc_auc_score(y_val_sample, p_val_sample[:, 1])
      }

def select_knn(n_neighbors, weights, algorithms, metrics, ps):
   combos = list(itertools.product(n_neighbors, weights, algorithms, metrics, ps))
   rows = Parallel(n_jobs=1)(delayed(evaluate_knn)(n, w, a, m, p) for n, w, a, m, p in combos)
   return pd.DataFrame(rows).sort_values(by="val_log_loss", ascending=True).reset_index(drop=True)

def evaluate_tree(criterion, max_depth, min_samples_leaf, splitter='best', min_samples_split=2, max_leaf_nodes=None, min_impurity_decrease=0.0):
   m = DecisionTreeClassifier(criterion=criterion, max_depth=max_depth, min_samples_leaf=min_samples_leaf, splitter=splitter, min_samples_split=min_samples_split, random_state=42)
   m.fit(X_train, y_train)
   p_train = m.predict_proba(X_train)
   p_val = m.predict_proba(X_val)
   return {
      "criterion": criterion,
      "splitter": splitter,
      "max_depth": max_depth,
      "min_samples_leaf": min_samples_leaf,
      "splits": min_samples_split,
      "train_log_loss": log_loss(y_train, p_train[:, 1]),
      "val_log_loss": log_loss(y_val, p_val[:, 1]),
      "train_auc": roc_auc_score(y_train, p_train[:, 1]),
      "val_auc": roc_auc_score(y_val, p_val[:, 1]),
      "max_leaf_nodes": max_leaf_nodes,
      "min_impurity_decrease": min_impurity_decrease
   }

def select_tree(criterions, max_depths, min_samples_leaves, splitters=['best', 'random'], min_samples_split=[2, 5, 10], max_leaf_nodes=None, min_impurity_decrease=0.0):
   combos = list(itertools.product(criterions, max_depths, min_samples_leaves, splitters, min_samples_split, max_leaf_nodes, min_impurity_decrease))
   rows = Parallel(n_jobs=-1)(delayed(evaluate_tree)(c, d, l, s, sp, ml, mi) for c, d, l, s, sp, ml, mi in combos)
   return pd.DataFrame(rows).sort_values(by="val_log_loss", ascending=True).reset_index(drop=True)

def evaluate_gaussian_nb(var_smoothing):
   m = GaussianNB(var_smoothing=var_smoothing)
   m.fit(X_train, y_train)
   p_train = m.predict_proba(X_train)
   p_val = m.predict_proba(X_val)
   return {
      "var_smoothing": var_smoothing,
      "train_log_loss": log_loss(y_train, p_train[:, 1]),
      "val_log_loss": log_loss(y_val, p_val[:, 1]),
      "train_auc": roc_auc_score(y_train, p_train[:, 1]),
      "val_auc": roc_auc_score(y_val, p_val[:, 1])
   }

def select_gaussian_nb(var_smoothing):
   rows = Parallel(n_jobs=-1)(delayed(evaluate_gaussian_nb)(v) for v in var_smoothing)
   return pd.DataFrame(rows).sort_values(by="val_log_loss", ascending=True).reset_index(drop=True)
    
def evaluate_random_forest(n_estimators, criterion, depth, min_samples_leaf, max_leaf_nodes, split=2,):
   m = RandomForestClassifier(n_estimators=n_estimators, criterion=criterion, max_depth=depth, min_samples_leaf=min_samples_leaf, max_leaf_nodes=max_leaf_nodes, min_samples_split=split, random_state=42, n_jobs=-1,)
   m.fit(X_train, y_train)
   p_train = m.predict_proba(X_train)
   p_val = m.predict_proba(X_val)
   return {
      "n_estimators": n_estimators,
      "criterion": criterion,
      "max_depth": depth,
      "min_samples_leaf": min_samples_leaf,
      "max_leaf_nodes": max_leaf_nodes,
      "splits": split,
      "train_log_loss": log_loss(y_train, p_train[:, 1]),
      "val_log_loss": log_loss(y_val, p_val[:, 1]),
      "train_auc": roc_auc_score(
         y_train, p_train[:, 1]
      ),
      "val_auc": roc_auc_score(
         y_val, p_val[:, 1]
      ),
    }

def select_random_forest(n_estimators, criterions, depths, min_samples_leaf, max_leaf_nodes, splits=[2, 5, 10],):
    combos = itertools.product(n_estimators, criterions, depths, min_samples_leaf, max_leaf_nodes, splits,)
    rows = Parallel(n_jobs=-1)(
        delayed(evaluate_random_forest)(n, c, d, mi, ma, sp)
        for n, c, d, mi, ma, sp in combos
    )
    return (
        pd.DataFrame(rows)
        .sort_values(by="val_auc", ascending=False)
        .reset_index(drop=True)
    )


# Majority Classifier

In [16]:
evaluate_model('Majority', DummyClassifier(strategy='most_frequent'), X_train, y_train, X_val, y_val)

{'accuracy': 0.5062100949779232,
 'confusion_matrix': [[0, 93270], [0, 95616]],
 'f1': 0.6721639918172807,
 'fit_time': 0.02510460000121384,
 'log_loss': 17.797992183660813,
 'model': 'Majority',
 'precision': 0.5062100949779232,
 'predict_time': 0.0009040000004461035,
 'recall': 1.0,
 'roc_auc': 0.5}


# Linear Regression

In [17]:
evaluate_model('Linear Regression', LinearRegression(), X_train, y_train, X_val, y_val)

{'accuracy': 0.5270533549336637,
 'confusion_matrix': [[43584, 49686], [39647, 55969]],
 'f1': 0.556155630965216,
 'fit_time': 0.9784799000008206,
 'log_loss': 0.6908403255373232,
 'model': 'Linear Regression',
 'precision': 0.529733566797596,
 'predict_time': 0.02054639999914798,
 'recall': 0.5853518239625167,
 'roc_auc': 0.5378091515753877}


# Logistic Regression

In [22]:
results = select_logistic_regression(
    solvers=['lbfgs', 'newton-cg', 'sag'],
    penalties=['l2', None],
    Cs=[0.00001, 0.0001, 0.001, 0.01]
)

print(results.head(5))

results = select_logistic_regression(
    solvers=['liblinear'],
    penalties=['l1', 'l2'],
    Cs=[0.00001, 0.0001, 0.0001, 0.001, 0.01]
)

print(results.head(5))

results = select_logistic_regression(
    solvers=['saga'],
    penalties=['l1', 'l2', 'elasticnet', None],
    Cs=[0.00001, 0.0001, 0.0001, 0.001, 0.01]
)

print(results.head(5))

      solver penalty       C  train_log_loss  val_log_loss  train_auc  \
0      lbfgs      l2  0.0001        0.689594      0.690824   0.548465   
1  newton-cg      l2  0.0001        0.689593      0.690828   0.548459   
2        sag      l2  0.0001        0.689593      0.690829   0.548458   
3  newton-cg      l2  0.0010        0.689571      0.690831   0.548451   
4        sag      l2  0.0010        0.689571      0.690832   0.548452   

    val_auc  
0  0.537722  
1  0.537700  
2  0.537699  
3  0.537846  
4  0.537845  
      solver penalty       C  train_log_loss  val_log_loss  train_auc  \
0  liblinear      l1  0.0100        0.689579      0.690825   0.548443   
1  liblinear      l2  0.0001        0.689593      0.690828   0.548458   
2  liblinear      l2  0.0001        0.689593      0.690828   0.548458   
3  liblinear      l2  0.0010        0.689571      0.690832   0.548452   
4  liblinear      l2  0.0100        0.689568      0.690839   0.548452   

    val_auc  
0  0.537821  
1  0.53769

In [23]:
evaluate_model("Logistic Regression", LogisticRegression(solver='lbfgs', penalty='l2', C=0.0001), X_train, y_train, X_val, y_val)

c:\Users\jacqu\Documents\crypto-predictor\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(


{'accuracy': 0.5273604184534587,
 'confusion_matrix': [[43264, 50006], [39269, 56347]],
 'f1': 0.5579767191994811,
 'fit_time': 0.8907723000011174,
 'log_loss': 0.6908243079297074,
 'model': 'Logistic Regression',
 'precision': 0.5298111007681965,
 'predict_time': 0.025377900001331,
 'recall': 0.5893051372155288,
 'roc_auc': 0.5377215134437898}


# KNN

In [ ]:
ks = [2200, 2300, 2400, 2500, 2600, 2800,]

t0 = time.perf_counter()
nn = NearestNeighbors(n_neighbors=max(ks), n_jobs=-1).fit(X_train_sample)
distances, indices = nn.kneighbors(X_val_sample)

neighbour_labels = y_train_sample[indices]

rows = []
for k in ks:
    scores = neighbour_labels[:, :k].mean(axis=1)
    y_pred = (scores > 0.5).astype(int)
    row = {
        'k': k,
        'accuracy': accuracy_score(y_val_sample, y_pred),
        'roc_auc_score': roc_auc_score(y_val_sample, scores),
        'log_loss': log_loss(y_val_sample, scores)
    }
    rows.append(row)
knn_results = pd.DataFrame(rows).sort_values(by="log_loss", ascending=True).reset_index(drop=True)
print(knn_results.head(10))

In [ ]:
results = select_knn(
    n_neighbors=[2500],
    weights=['uniform', 'distance'],
    algorithms=['auto', 'ball_tree', 'kd_tree', 'brute'],
    metrics=['euclidean', 'manhattan', 'minkowski'],
    ps=[1, 2]
)

print(results.head(5))

X_train: (1127670, 36)
y_train: (1127670,)
p_train: (200000, 2)
X_val: (188886, 36)
y_val: (188886,)
p_val: (30000, 2)
X_train: (1127670, 36)
y_train: (1127670,)
p_train: (200000, 2)
X_val: (188886, 36)
y_val: (188886,)
p_val: (30000, 2)


In [13]:
evaluate_model('3501-NN', KNeighborsClassifier(n_neighbors=3501, n_jobs=-1), X_train_sample, y_train_sample, X_val_sample, y_val_sample)

{'accuracy': 0.5305,
 'confusion_matrix': [[7144, 7585], [6500, 8771]],
 'f1': 0.5546526701868657,
 'fit_time': 0.030631000001449138,
 'model': '3501-NN',
 'precision': 0.5362558082660798,
 'predict_time': 20.106372700000065,
 'recall': 0.5743566236657717,
 'roc_auc': 0.5424871546627803}


# Decision Tree

In [ ]:
results = select_tree(
   criterions=['entropy', 'gini', 'log_loss'],
   max_depths=[4, 5, 6],
   min_samples_leaves=[2000, 3000],
   splitters=['best', 'random'],
   min_samples_split=[2, 24, 48, 200, 10000],
   max_leaf_nodes=[None],
   min_impurity_decrease=[0.0, 0.1]
   )

print(results.head(10))

  criterion splitter  max_depth  min_samples_leaf  splits  train_log_loss  \
0   entropy     best          4             20000      48        0.689068   
1   entropy     best          4             20000   10000        0.689068   
2   entropy     best          4             20000  200000        0.689559   

   val_log_loss  train_auc   val_auc max_leaf_nodes  min_impurity_decrease  
0      0.691676   0.550431  0.532460           None                    0.0  
1      0.691676   0.550431  0.532460           None                    0.0  
2      0.692232   0.546974  0.528895           None                    0.0  


In [16]:
evaluate_model('ID3', DecisionTreeClassifier(criterion='entropy', splitter='best', max_depth=4, min_samples_leaf=20000, min_samples_split=48, max_leaf_nodes=None, min_impurity_decrease=0.0, random_state=42), X_train, y_train, X_val, y_val)

{'accuracy': 0.522902703217814,
 'confusion_matrix': [[51043, 42227], [47890, 47726]],
 'f1': 0.5143747069823085,
 'fit_time': 14.560612000001129,
 'log_loss': 0.6916756747055899,
 'model': 'ID3',
 'precision': 0.5305659622247173,
 'predict_time': 0.03361589999985881,
 'recall': 0.4991424029451138,
 'roc_auc': 0.5324595413568789}


# Random Forest

In [ ]:
results = select_random_forest(
    n_estimators=[10, 50, 100],
    criterions=['entropy', 'gini'],
    depths=[3, 5, 7],
    min_samples_leaf=[1, 2, 5],
    max_leaf_nodes=[10, 20, 50],
    splits=[2]
)

print(results)

# Naive Bayes

In [23]:
results = select_gaussian_nb(
    var_smoothing=[1e-9, 1e-8, 1e-7, 1e-6, 1e-5, 1e-4, 1e-3, 1e-2, 1e-1, 1, 10, 20, 50, 100, 200]
)

print(results)

    var_smoothing  train_log_loss  val_log_loss  train_auc   val_auc
0    1.000000e+01        0.691586      0.692103   0.546570  0.535133
1    2.000000e+01        0.692130      0.692422   0.546591  0.534966
2    5.000000e+01        0.692634      0.692766   0.546564  0.534894
3    1.000000e+02        0.692834      0.692910   0.546550  0.534880
4    2.000000e+02        0.692941      0.692989   0.546542  0.534876
5    1.000000e+00        0.712818      0.705514   0.544433  0.534740
6    1.000000e-01        0.802785      0.757545   0.541794  0.532754
7    1.000000e-02        0.829269      0.773140   0.541303  0.532313
8    1.000000e-03        0.832453      0.775202   0.541113  0.532172
9    1.000000e-04        0.832778      0.775416   0.541088  0.532153
10   1.000000e-05        0.832810      0.775438   0.541086  0.532151
11   1.000000e-06        0.832813      0.775440   0.541086  0.532151
12   1.000000e-07        0.832814      0.775440   0.541085  0.532151
13   1.000000e-08        0.832814 

In [22]:
evaluate_model("Naive Bayes", GaussianNB(var_smoothing=10), X_train, y_train, X_val, y_val)

{'accuracy': 0.5162743665491355,
 'confusion_matrix': [[72061, 21209], [70160, 25456]],
 'f1': 0.35782711676190077,
 'fit_time': 0.5746155999950133,
 'log_loss': 0.6921025471995634,
 'model': 'Naive Bayes',
 'precision': 0.5455051966141647,
 'predict_time': 0.13139360000059241,
 'recall': 0.26623159303882193,
 'roc_auc': 0.5351330642429624}
